# Scrye — Experiments Notebook (compare simulation systems & models)

The lab bench for Part I. Where `00_starter.ipynb` runs **one** pipeline end-to-end,
this notebook **compares many** on identical data and reports topline numbers, so you
can iterate on the two things that matter:

- **the simulation system** — which `Predictor` (+ optional `Calibrator`) is inside the pipeline
- **the model** — which OpenRouter model it calls (one variable, swapped freely)

### Holdout discipline (read once)

Every design decision here is *fit to data* even though nothing is gradient-trained, so we
split **by question family** (all segment variants of a survey item move together — no leakage)
into `dev` / `val` / `test`:

| bucket | role | touched |
|---|---|---|
| **dev** | iterate, compare systems, read errors | freely (this notebook) |
| **val** | model/hyperparameter selection, fit calibration params | for selection |
| **test** | final reported numbers; **required questions pinned here** | **once, at the end** |

All comparison below runs on **dev**. `test` stays sealed.

In [ ]:
%load_ext autoreload
from IPython import get_ipython
%autoreload 2

from scrye.config import DEFAULT_MODEL, OPENROUTER_MODELS, FIGURES_DIR
from scrye.data import load_all
from scrye.splits import make_split
from scrye import evaluate as ev
from scrye import experiment as xp
from scrye import viz

import io
from matplotlib.figure import Figure

def _figure_to_png(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    return buf.getvalue()

get_ipython().display_formatter.formatters["image/png"].for_type(Figure, _figure_to_png)


In [ ]:

recs = load_all()
all_records = recs["pop"] + recs["grouped"]

# Per-dataset SimBench Eq. 2 normalizers, built from the FULL data so scores are
# comparable across systems, subsamples, and the paper — independent of the split.
normalizers = ev.build_normalizers(all_records)
print(f"loaded {len(all_records)} records; {len(normalizers)} dataset normalizers")

## 1. Build the split

Leave-family-out, deterministic and seed-controlled. The summary shows family/record counts
per bucket and confirms the three required questions are pinned to `test`.

In [ ]:
split = make_split(all_records, seed=0)   # default fractions dev .50 / val .25 / test .25
split.check_disjoint()                    # raises if any family straddles buckets
split.summary()

## 2. Knobs — set the model and the systems to compare  ← edit here

`MODEL` accepts a short key from `OPENROUTER_MODELS` (e.g. `"qwen-72b"`) or a full model id.
`SYSTEMS` maps a label → a pipeline; add a row to compare a new simulation system. New
predictors register in `scrye.experiment.PREDICTOR_REGISTRY` and become selectable by name.

In [ ]:
print("available model keys:", list(OPENROUTER_MODELS))

MODEL = "gemini-flash-lite"      # <-- the swappable model variable
N_DEV = 120                      # dev evaluation subsample size (bump for tighter CIs)

# The dev evaluation set: dataset-stratified subsample of the DEV bucket only.
dev_eval = ev.stratified_sample(split.dev, n=N_DEV, seed=0)
print(f"dev eval sample: {len(dev_eval)} records "
      f"across {len({r.dataset_name for r in dev_eval})} datasets")

# The systems under comparison. Each is a Pipeline; swap predictor/calibrator/model freely.
SYSTEMS = {
    "uniform (floor)":      xp.build_pipeline(predictor="uniform"),
    f"zero-shot @ {MODEL}": xp.build_pipeline(model=MODEL, predictor="zero_shot"),
}

## 3. Run the comparison → topline table

In [ ]:
topline = xp.compare(SYSTEMS, dev_eval, normalizers=normalizers)
topline

In [ ]:
viz.plot_system_comparison(topline)

## 4. Model sweep — the portability ablation

Hold the simulation system fixed, vary only the model. Thesis: raw zero-shot tracks model
capability, but a *method's gain* should be roughly model-invariant. Add model keys to the
list to widen the sweep (each new/uncached model spends a little API budget).

In [ ]:
SWEEP_MODELS = ["gemini-flash-lite", "qwen-7b"]   # <-- add models to compare

sweep = xp.model_sweep(SWEEP_MODELS, dev_eval, predictor="zero_shot", normalizers=normalizers)
sweep

In [ ]:
viz.plot_system_comparison(sweep, label_col="system")

## 5. Drill into one system

`compare()` stashes each system's full per-record result frame under `.attrs["results"]`,
so the whole `viz` suite from `00_starter.ipynb` works on any compared system — e.g. the
mode-seeking test (does it fail on high-entropy / divided questions?).

In [ ]:
label = f"zero-shot @ {MODEL}"
res = topline.attrs["results"][label]
print("per-record rows:", len(res))
viz.plot_score_vs_entropy(res)

## 6. Discipline reminder

- Everything above ran on **dev**. Iterate here as much as you like.
- Use **val** to pick the winning system/model and fit any calibration parameters.
- Score **test** exactly once, at the end, for the reported numbers — it holds the
  required questions and must never inform a choice made above.

```python
# final, run-once:
# test_eval = ev.stratified_sample(split.test, n=400, seed=0)
# xp.compare({"final system": best_pipeline}, test_eval, normalizers=normalizers)
```